In [ ]:
import cudf
import xgboost as xgb
import numpy as np

print("🚀 Loading 1,000,000 rows into GPU VRAM...")
gdf = cudf.read_csv('09_booking_capacity_data.csv')

# 1. Define our Features (X) and our Target (y)
X = gdf[['days_to_departure', 'waitlist_depth', 'surge_factor', 'historical_cancel_rate']]
y = gdf['is_confirmed']

# 2. Split into Training (80%) and Testing (20%) data natively on the GPU
split_idx = int(len(gdf) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("🧠 Training XGBoost Model on RTX 3050 Ti...")
# Convert to DMatrix (XGBoost's highly optimized data structure)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# The Magic Parameters: This tells XGBoost to bypass the CPU entirely
params = {
    'objective': 'binary:logistic',
    'tree_method': 'hist',
    'device': 'cuda',  # <--- THIS IS THE CUDA TRIGGER
    'eval_metric': 'logloss'
}

# 3. Train the Model!
model = xgb.train(params, dtrain, num_boost_round=100)
print("✅ Training Complete!")

# 4. Test the model's accuracy on the 20% of data it has never seen before
preds = model.predict(dtest)
predictions = (preds > 0.5).astype(int)

# Compare predictions to actual outcomes
y_test_np = y_test.to_numpy()
accuracy = (predictions == y_test_np).mean()
print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")

🚀 Loading 1,000,000 rows into GPU VRAM...
🧠 Training XGBoost Model on RTX 3050 Ti...
✅ Training Complete!
🎯 Model Accuracy: 84.63%


In [2]:
import pandas as pd

print("🔮 Running AI Inference on New Scenarios...")

# Scenario 1: The "Diwali Rush" - Booking only 10 days out, massive waitlist (150), high surge (2.5)
# Scenario 2: The "Smart Planner" - Booking 45 days out, tiny waitlist (12), normal surge (1.0)
inference_data = pd.DataFrame({
    'days_to_departure': [10, 45],
    'waitlist_depth': [150, 12],
    'surge_factor': [2.5, 1.0], 
    'historical_cancel_rate': [0.15, 0.25]
})

# Convert to XGBoost DMatrix
d_inference = xgb.DMatrix(inference_data)

# Predict the probability of confirmation (returns a decimal between 0 and 1)
probabilities = model.predict(d_inference)

print("\n--- AI PREDICTIONS ---")
print(f"🚨 Scenario 1 (Holiday Rush) Confirmation Chance: {probabilities[0] * 100:.1f}%")
print(f"✅ Scenario 2 (Smart Planner) Confirmation Chance: {probabilities[1] * 100:.1f}%")

🔮 Running AI Inference on New Scenarios...

--- AI PREDICTIONS ---
🚨 Scenario 1 (Holiday Rush) Confirmation Chance: 2.8%
✅ Scenario 2 (Smart Planner) Confirmation Chance: 78.6%


In [ ]:
# Save the trained model to a file
model.save_model("09_capacity_model.json")
print("✅ Model successfully saved to disk as '09_capacity_model.json'")

✅ Model successfully saved to disk as 'capacity_model.json'
